# Module 3.2 — Dynamic Metadata Extraction (Gemini Batch API Edition)

**Kiến trúc:**
- 🔄 **Inference**: Gemini Batch API — gửi 10 doc/batch, xử lý song song, **giảm 50% chi phí**
- ✅ **Context Cache**: Cache system prompt + schema 1 lần cho toàn session, **giảm ~75% input token**
- ✅ **Retry thông minh**: Batch fail → fallback sang direct API, không mất dữ liệu
- ✅ **Token budget**: Ước lượng token trước khi gửi, tự cắt text để không vượt giới hạn output

| Cơ chế | Tiết kiệm | Ghi chú |
|---|---|---|
| Batch API | −50% chi phí | Async, kết quả trong 1–24h |
| Context Cache | −75% input token phần system | Cache schema cố định |
| Smart truncate | −20–40% input token phần văn bản | Giữ phần quan trọng |
| max_output_tokens=512 | Khống chế output | JSON schema nhỏ, không cần 1024 |

In [13]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [14]:
!pip install -q pydantic json-repair google-genai tqdm

In [15]:
import json
import logging
import re
import time
import math
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple, Type
from concurrent.futures import ThreadPoolExecutor, as_completed
from google import genai
from google.genai import types
from json_repair import repair_json
from pydantic import BaseModel, Field
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("UIE")
# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
@dataclass
class ExtractionConfig:
    model_name: str   = "gemini-2.0-flash"

    # ── Token budget ──────────────────────────────────────────────────────────
    max_output_tokens: int   = 512      # JSON output thực tế < 300 token → 512 dư
    max_input_chars: int     = 3000     # ~750 token input văn bản mỗi doc
    max_input_chars_retry: int = 1500   # Cắt ngắn hơn khi retry

    # ── Batch config ──────────────────────────────────────────────────────────
    batch_size: int          = 20       # Số doc mỗi batch
    batch_poll_interval: int = 5       # Giây chờ giữa mỗi lần poll
    batch_timeout_sec: int   = 300     # 2h timeout tổng

    # ── Retry ─────────────────────────────────────────────────────────────────
    max_retries: int         = 2        # Retry direct API khi batch fail
    retry_delay: float       = 1.5      # Giây backoff giữa retry

    # ── Confidence ────────────────────────────────────────────────────────────
    low_confidence_threshold: float = 0.65
    min_fill_rate_warn: float = 0.40
    # ── Paths ─────────────────────────────────────────────────────────────────
    input_file:  str = "/content/drive/MyDrive/VNDigitize/Inputdata.json"
    output_file: str = "/content/drive/MyDrive/VNDigitize/UIE_Extraction_Results_batch.json"

CFG = ExtractionConfig()

# ── API Key ───────────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    _api_key = userdata.get("llmextration") # Changed from what looked like a key to a secret name
    logger.info("✅ API key từ Colab Secrets")
except Exception:
    _api_key = ""
    logger.warning("⚠️  Chưa có API key — điền _api_key thủ công")

client = genai.Client(api_key=_api_key)
logger.info(f"Model: {CFG.model_name} | Batch size: {CFG.batch_size} | Max output tokens: {CFG.max_output_tokens}")

In [16]:
# ══════════════════════════════════════════════════════════════════════════════
# SCHEMA DEFINITIONS
# ══════════════════════════════════════════════════════════════════════════════
from pydantic import BaseModel, Field
from typing import Optional, Dict, Type, List

class VanBanToaAnSchema(BaseModel):
    loai_van_ban:              Optional[str] = Field(default=None, description="VD: Bản án sơ thẩm, Quyết định công nhận thuận tình ly hôn")
    so_hieu:                   Optional[str] = Field(default=None, description="VD: 03/2022/DSST, 89/2026/QĐST-HNGĐ")
    ngay_ban_hanh:             Optional[str] = Field(default=None, description="VD: 23/11/2022")
    ten_toa_an:                Optional[str] = Field(default=None, description="VD: Tòa án nhân dân tỉnh BN")
    vu_viec_tom_tat:           Optional[str] = Field(default=None, description="Tóm tắt nội dung vụ việc")
    tham_phan_chu_toa:         Optional[str] = Field(default=None, description="Tên Thẩm phán hoặc Chủ tọa")
    nguyen_don_nguoi_khoi_kien:Optional[str] = Field(default=None, description="Tên Nguyên đơn / Người khởi kiện / Người yêu cầu")
    bi_don_nguoi_bi_kien:      Optional[str] = Field(default=None, description="Tên Bị đơn / Người bị kiện / Bị cáo")

class VanBanHanhChinhSchema(BaseModel):
    loai_van_ban:    Optional[str] = Field(default=None, description="VD: Quyết định, Thông báo, Kết luận thanh tra")
    so_hieu:         Optional[str] = Field(default=None, description="VD: 1059/QĐ-UBND")
    ngay_ban_hanh:   Optional[str] = Field(default=None, description="VD: 29 tháng 3 năm 2021")
    co_quan_ban_hanh:Optional[str] = Field(default=None, description="VD: UBND thành phố Đà Lạt")
    noi_dung_tom_tat:Optional[str] = Field(default=None, description="Về việc gì")
    nguoi_ky:        Optional[str] = Field(default=None, description="Tên người ký văn bản")

class BienBanSchema(BaseModel):
    ten_bien_ban:            Optional[str] = Field(default=None, description="VD: Biên bản ghi lời khai, vi phạm hành chính")
    thoi_gian_lap:           Optional[str] = Field(default=None, description="Thời gian lập biên bản")
    dia_diem_lap:            Optional[str] = Field(default=None, description="Nơi lập biên bản")
    nguoi_chu_tri_lap:       Optional[str] = Field(default=None, description="Tên cán bộ / người lập")
    nguoi_tham_gia_doi_tuong:Optional[str] = Field(default=None, description="Tên người được lấy lời khai / bị lập biên bản")
    noi_dung_chinh:          Optional[str] = Field(default=None, description="Tóm tắt nội dung biên bản")

class VanBanDieuTraSchema(BaseModel):
    loai_van_ban:       Optional[str] = Field(default=None, description="VD: Bản kết luận điều tra, Lệnh cấm đi khỏi nơi cư trú")
    so_hieu:            Optional[str] = Field(default=None, description="VD: 65/KLĐT-PC03")
    ngay_ban_hanh:      Optional[str] = Field(default=None, description="VD: 03 tháng 11 năm 2021")
    co_quan_ban_hanh:   Optional[str] = Field(default=None, description="VD: Cơ quan Cảnh sát điều tra Công an tỉnh Quảng Bình")
    ten_bi_can_doi_tuong:Optional[str] = Field(default=None, description="Họ tên bị can / đối tượng bị điều tra")
    toi_danh_vu_an:     Optional[str] = Field(default=None, description="Tội danh hoặc tên vụ án")

class DonTuCamKetSchema(BaseModel):
    loai_giay_to:              Optional[str] = Field(default=None, description="VD: Đơn tố cáo, Bản cam kết, Giấy xác nhận")
    nguoi_viet_don_cam_ket:    Optional[str] = Field(default=None, description="Tên người đứng đơn")
    noi_nhan_co_quan_giai_quyet:Optional[str]= Field(default=None, description="Kính gửi ai / cơ quan nào")
    ngay_thang_nam:            Optional[str] = Field(default=None, description="Ngày tháng ghi trên đơn")
    noi_dung_tom_tat:          Optional[str] = Field(default=None, description="Tóm tắt nội dung")
class VanBanBaoHiemSchema(BaseModel):
    loai_van_ban:         Optional[str] = Field(default=None, description="VD: Quyết định hưởng trợ cấp thất nghiệp, Thông báo đóng BHXH, Thẻ bảo hiểm")
    so_hieu:              Optional[str] = Field(default=None, description="VD: 1452/QĐ-BHXH, 4523/TB-BHXH")
    ngay_ban_hanh:        Optional[str] = Field(default=None, description="VD: 20/05/2026")
    co_quan_ban_hanh:     Optional[str] = Field(default=None, description="VD: Bảo hiểm xã hội quận Cầu Giấy, BHXH TP. Hồ Chí Minh")
    nguoi_duoc_bao_hiem:  Optional[str] = Field(default=None, description="Họ và tên người tham gia / người được hưởng bảo hiểm")
    ma_so_bhxh_the_bhyt:  Optional[str] = Field(default=None, description="Mã số BHXH hoặc Số thẻ BHYT (nếu có)")
    noi_dung_tom_tat:     Optional[str] = Field(default=None, description="Tóm tắt nội dung bảo hiểm (VD: hưởng trợ cấp 3 tháng, thông báo nợ tiền đóng)")
class HopDongSchema(BaseModel):
    loai_hop_dong:     Optional[str] = Field(default=None, description="VD: Hợp đồng thế chấp, Hợp đồng mua bán, Hợp đồng vay")
    so_hop_dong:       Optional[str] = Field(default=None, description="VD: 4552/HĐTC-VPB")
    ngay_ky:           Optional[str] = Field(default=None, description="Ngày ký hợp đồng")
    ben_a:             Optional[str] = Field(default=None, description="Bên A / Bên cho vay / Bên bán / Bên nhận thế chấp")
    ben_b:             Optional[str] = Field(default=None, description="Bên B / Bên vay / Bên mua / Bên thế chấp")
    tai_san_doi_tuong: Optional[str] = Field(default=None, description="Tài sản thế chấp / đối tượng hợp đồng")
    noi_dung_tom_tat:  Optional[str] = Field(default=None, description="Tóm tắt nội dung hợp đồng")
    nguoi_chung_thuc:  Optional[str] = Field(default=None, description="Công chứng viên / người chứng thực")
SCHEMA_REGISTRY: Dict[str, Type[BaseModel]] = {
    "VAN_BAN_TOA_AN":       VanBanToaAnSchema,
    "VAN_BAN_DIEU_TRA":     VanBanDieuTraSchema,
    "QUYET_DINH_HANH_CHINH":VanBanHanhChinhSchema,
    "BIEN_BAN":             BienBanSchema,
    "DON_TU_CAM_KET":       DonTuCamKetSchema,
    "VAN_BAN_BAO_HIEM":     VanBanBaoHiemSchema,
    "HOP_DONG":              HopDongSchema,
}

SCHEMA_KEYWORDS: List[tuple] = [
    ("VAN_BAN_TOA_AN",        ["tòa án nhân dân","bản án số","quyết định đình chỉ xét xử","tòa phúc thẩm","tòa án","thẩm phán","xét xử"]),
    ("VAN_BAN_DIEU_TRA",      ["kết luận điều tra","cơ quan cảnh sát điều tra","viện kiểm sát","khởi tố bị can","lệnh cấm đi khỏi","truy tố","tội phạm"]),
    ("BIEN_BAN",              ["biên bản vi phạm","biên bản ghi lời khai","biên bản đối chất","biên bản thỏa thuận","biên bản giao nhận"]),
    ("QUYET_DINH_HANH_CHINH", ["quyết định","ủy ban nhân dân","ubnd","thanh tra tỉnh","thu hồi","sở tài nguyên","đình chỉ công tác"]),
    ("VAN_BAN_BAO_HIEM",      ["bảo hiểm xã hội","bhxh","bảo hiểm y tế","bhyt","trợ cấp thất nghiệp","sổ bảo hiểm"]),
    ("HOP_DONG",              ["hợp đồng", "bên a", "bên b", "công chứng", "thế chấp", "bên vay", "bên cho vay", "văn phòng công chứng"]),
    ("DON_TU_CAM_KET",        ["đơn tố cáo","bản cam kết","giấy cam kết","thư cảm ơn","giấy xác nhận","đơn khiếu nại","tôi tên là"]),
]

def auto_detect_schema(text: str) -> str:
    """
    Detect schema theo 3 pass:
    - Pass 0: nhận diện tiêu đề/dòng đầu (ưu tiên cao nhất, ít bị nhiễu)
    - Pass 1: keyword scoring toàn văn (tính hits, lấy max)
    - Pass 2: regex fallback khi không đủ tín hiệu
    """
    t = text.lower()
    # Chỉ lấy ~300 ký tự đầu để detect tiêu đề, tránh bị từ trong nội dung nhiễu
    header = t[:300]

    # ── Pass 0: match tiêu đề trực tiếp (độ chính xác cao nhất) ─────────────
    _HEADER_RULES = [
        ("VAN_BAN_BAO_HIEM",      [r"(bhxh|bhyt|bảo\s*hiểm\s*(xã\s*hội|y\s*tế)|thẻ\s*bảo\s*hiểm|trợ\s*cấp\s*thất\s*nghiệp)"]),
        ("HOP_DONG",              [r"(hợp\s*đồng\s*(thế\s*chấp|mua\s*bán|vay|dịch\s*vụ|lao\s*động)|văn\s*phòng\s*công\s*chứng)"]),
        ("BIEN_BAN",              [r"biên\s*bản\s*(vi\s*phạm|ghi\s*lời|đối\s*chất|khám\s*nghiệm|thỏa\s*thuận|giao\s*nhận)"]),
        ("VAN_BAN_DIEU_TRA",      [r"(bản\s*kết\s*luận\s*điều\s*tra|cáo\s*trạng|lệnh\s*(bắt|tạm\s*giam|khám\s*xét))"]),
        ("VAN_BAN_TOA_AN",        [r"(bản\s*án\s*số|nhân\s*danh\s*nước\s*cộng\s*hòa|tòa\s*(phúc\s*thẩm|án\s*nhân\s*dân))"]),
        ("QUYET_DINH_HANH_CHINH", [r"(quyết\s*định|kết\s*luận\s*thanh\s*tra|giấy\s*phép\s*xây\s*dựng)"]),
        ("DON_TU_CAM_KET",        [r"(đơn\s*(khiếu\s*nại|tố\s*cáo|yêu\s*cầu)|bản\s*cam\s*kết|tôi\s+tên\s+là)"]),
    ]
    for schema_name, patterns in _HEADER_RULES:
        for pat in patterns:
            if re.search(pat, header, re.IGNORECASE):
                return schema_name

    # ── Pass 1: keyword scoring toàn văn ─────────────────────────────────────
    scores: Dict[str, int] = {}
    for name, kws in SCHEMA_KEYWORDS:
        hits = sum(1 for k in kws if k in t)
        if hits > 0:
            scores[name] = hits

    if scores:
        best_name  = max(scores, key=lambda n: scores[n])
        best_score = scores[best_name]

        if best_score >= 2:
            return best_name

        top = [n for n, s in scores.items() if s == best_score]
        if len(top) == 1:
            return best_name
        # Tie → theo thứ tự ưu tiên
        for name, _ in SCHEMA_KEYWORDS:
            if name in top:
                return name

    # ── Pass 2: regex fallback ────────────────────────────────────────────────
    if re.search(r"bản\s+án|xét\s+xử|tuyên\s+phạt", t, re.IGNORECASE):
        return "VAN_BAN_TOA_AN"
    if re.search(r"điều\s+tra|khởi\s+tố|truy\s+tố", t, re.IGNORECASE):
        return "VAN_BAN_DIEU_TRA"
    if re.search(r"biên\s+bản", t, re.IGNORECASE):
        return "BIEN_BAN"
    if re.search(r"bhxh|bhyt|bảo\s+hiểm", t, re.IGNORECASE):
        return "VAN_BAN_BAO_HIEM"
    if re.search(r"\d{1,4}/\d{4}/[A-ZĐĂÂÊÔƯĐ]{2,}", t):
        return "QUYET_DINH_HANH_CHINH"

    return "DON_TU_CAM_KET"

In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# PROMPT BUILDER + JSON PARSER + CONFIDENCE SCORER
# ══════════════════════════════════════════════════════════════════════════════

# ── Truncate ──────────────────────────────────────────────────────────────────
_KEY_PATS = [
    re.compile(r"(?:bị đơn|nguyên đơn|bị cáo)[:\s].+", re.IGNORECASE),
    re.compile(r"\d{1,3}/\d{4}/[A-Z]{2,}"),
    re.compile(r"ngày\s+\d{1,2}\s+tháng", re.IGNORECASE),
    re.compile(r"(?:thẩm phán|kiểm sát viên|điều tra viên)[:\s].+", re.IGNORECASE),
]

def smart_truncate(text: str, max_chars: int) -> str:
    if len(text) <= max_chars:
        return text
    cut = int(max_chars * 0.78)
    head = text[:cut]
    tail = text[cut:]
    key_lines = [
        ln.strip() for ln in tail.splitlines()
        if ln.strip() and any(p.search(ln) for p in _KEY_PATS)
    ]
    snippet = " | ".join(key_lines)[:int(max_chars * 0.22)]
    return head + ("\n...[cắt]...\n" + snippet if snippet else " ...[cắt]")

# ── Prompt builder ────────────────────────────────────────────────────────────
def _field_list(sc: Type[BaseModel]) -> str:
    props = sc.model_json_schema().get("properties", {})
    return "\n".join(f"- {k}: {v.get('description','')}" for k, v in props.items())

def _empty_template(sc: Type[BaseModel]) -> str:
    props = sc.model_json_schema().get("properties", {})
    template = {k: None for k in props}
    template["summary"] = None    # ← thêm dòng này
    return json.dumps(template, ensure_ascii=False, indent=2)

_SYSTEM_HEADER = (
    "Bạn là chuyên gia trích xuất dữ liệu văn bản pháp lý Việt Nam.\n"
    "Nhiệm vụ: đọc văn bản và trả về JSON với các trường cho sẵn.\n"
    "Quy tắc QUAN TRỌNG:\n"
    "1. Chỉ trả về JSON thuần tuý — không markdown, không giải thích.\n"
    "2. Không tìm thấy → null (không dùng chuỗi rỗng).\n"
    "3. Nhiều người → nối bằng '; '.\n"
    "4. Giữ nguyên key, chỉ điền value.\n"
    "5. Luôn thêm key 'summary': tóm tắt nội dung văn bản trong 2–3 câu tiếng Việt.\n"

)

def build_prompt(schema_class: Type[BaseModel], raw_text: str, retry: bool = False) -> str:
    max_c = CFG.max_input_chars_retry if retry else CFG.max_input_chars
    text  = smart_truncate(raw_text, max_c)

    # Detect nếu text có vẻ là multi-page (có dấu hiệu page break)
    is_multipage = bool(
        re.search(r"\n{3,}|={5,}|─{5,}|\[trang\s*\d+\]|page\s*\d+", text, re.IGNORECASE)
    )
    multipage_hint = (
        "\nLƯU Ý: Văn bản gồm nhiều trang — thông tin như số hiệu, ngày, tên cơ quan "
        "có thể xuất hiện ở trang sau, không nhất thiết ở đầu văn bản. "
        "Hãy đọc toàn bộ và trích xuất dù thông tin nằm ở bất kỳ trang nào.\n"
        if is_multipage else ""
    )

    if retry:
            keys = list(schema_class.model_json_schema().get("properties", {}).keys())
            content_page_hint = (
                "\nVăn bản này có thể là trang nội dung (thiếu tiêu đề/số hiệu ở đầu). "
                "Hãy suy luận từ các dấu hiệu trong nội dung: danh sách đương sự, "
                "tên tranh chấp, cơ quan được nhắc đến... để điền các trường còn thiếu. "
                "Nếu thực sự không có thông tin → null.\n"
            )
            return (
                f"Trích xuất {', '.join(keys)} từ văn bản pháp lý tiếng Việt.\n"
                f"{multipage_hint}"
                f"{content_page_hint}"
                "Trả về JSON thuần tuý. Không tìm thấy → null.\n\n"
                f"VĂN BẢN:\n{text}"
            )
    return (
        f"{_SYSTEM_HEADER}"
        f"{multipage_hint}"
        f"\nTRƯỜNG CẦN TRÍCH XUẤT:\n{_field_list(schema_class)}\n\n"
        f"TEMPLATE:\n{_empty_template(schema_class)}\n\n"
        f"VĂN BẢN:\n{text}"
    )

# ── JSON Parser ───────────────────────────────────────────────────────────────
def clean_and_parse(raw: str, doc_id: str = "") -> Optional[dict]:
    raw = raw.strip()
    m = re.search(r"```(?:json)?\s*([\s\S]+?)```", raw)
    candidate = m.group(1).strip() if m else raw
    s, e = candidate.find("{"), candidate.rfind("}")
    if s != -1 and e != -1:
        candidate = candidate[s:e+1]
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        pass
    try:
        r = repair_json(candidate, return_objects=True)
        if isinstance(r, dict) and r:
            return r
    except Exception:
        pass
    if doc_id:
        logger.warning(f"[{doc_id}] parse fail | raw[:200]={repr(raw[:200])}")
    return None

# ── Confidence Scoring ────────────────────────────────────────────────────────
_RE_SO_HIEU  = re.compile(
    r"^\d{1,4}"           # số thứ tự: 1–4 chữ số
    r"[\s/–\-]+"          # dấu phân cách (/, –, -, khoảng trắng)
    r"\d{4}"              # năm 4 chữ số
    r"[\s/–\-]+"
    r"[A-ZĐÂÊÔĂƯĐ]{2,}"  # ký hiệu cơ quan
)
_RE_DATE_VN  = re.compile(r"\d{1,2}\s*[/\-\.]\s*\d{1,2}\s*[/\-\.]\s*\d{4}")
_RE_DATE_TXT = re.compile(r"\d{1,2}\s+tháng\s+\d{1,2}\s+năm\s+\d{4}", re.IGNORECASE)
_RE_DATE_DASH= re.compile(r"\d{1,2}\s*-\s*\d{1,2}\s*-\s*\d{4}")   # format 29 - 9 - 2025
_RE_CCCD     = re.compile(r"^\d{9}$|^\d{12}$")
_RE_DIGIT    = re.compile(r"\d")
_RE_TIME_VN  = re.compile(r"\d{1,2}\s*giờ.*ngày\s*\d{1,2}", re.IGNORECASE)  # 09 giờ 30 phút ngày...

def compute_field_confidence(value: Any, fname: str) -> float:
    if value is None or str(value).strip() == "":
        return 0.0
    v = str(value).strip()
    f = fname.lower()

    # ── so_hieu ──────────────────────────────────────────────────────────────
    if any(k in f for k in ("so_hieu", "so_ban_an", "thu_ly", "so_hop_dong")):
        if _RE_SO_HIEU.match(v):
            return 0.95
        # có số + chữ hoa nhưng format không chuẩn
        if _RE_DIGIT.search(v) and re.search(r"[A-ZĐĂÂÊÔƯĐ]{2,}", v):
            return 0.80
        return 0.72 if _RE_DIGIT.search(v) else 0.45

    # ── ngay_ban_hanh / thoi_gian_lap ────────────────────────────────────────
    if any(k in f for k in ("ngay", "ngày", "date", "thoi_gian", "thời_gian")):
        # Datetime đầy đủ: có giờ + ngày
        if _RE_TIME_VN.search(v):
            return 0.90
        # Ngày đầy đủ các format
        if _RE_DATE_TXT.search(v) or _RE_DATE_VN.search(v) or _RE_DATE_DASH.search(v):
            return 0.92
        # Chỉ có số, có thể là ngày viết tắt
        if _RE_DIGIT.search(v):
            return 0.55
        return 0.30

    # ── năm sinh ─────────────────────────────────────────────────────────────
    if any(k in f for k in ("nam_sinh", "year")):
        try:
            return 0.95 if 1900 <= int(v) <= 2030 else 0.50
        except:
            return 0.30

    # ── CCCD / CMND ──────────────────────────────────────────────────────────
    if any(k in f for k in ("cccd", "cmnd")):
        return 0.95 if _RE_CCCD.match(v.replace(" ", "")) else (0.65 if _RE_DIGIT.search(v) else 0.35)

    # ── mã số BHXH / thẻ BHYT ───────────────────────────────────────────────
    if any(k in f for k in ("ma_so_bhxh", "the_bhyt", "ma_so")):
        clean = v.replace(" ", "")
        # BHXH: 10 chữ số; BHYT: bắt đầu bằng 2 chữ cái + 13 chữ số
        if re.match(r"^\d{10}$", clean):
            return 0.95
        if re.match(r"^[A-Z]{2}\d{13}$", clean):
            return 0.95
        return 0.65 if _RE_DIGIT.search(v) else 0.35

    # ── tên người ────────────────────────────────────────────────────────────
    _NAME_KWS = ("ho_ten", "tham_phan", "thu_ky", "kiem_sat", "ten_bi",
                 "nguyen_don", "bi_don", "nguoi_", "chu_tri", "nguoi_viet",
                 "nguoi_duoc", "nguoi_ky")
    if any(k in f for k in _NAME_KWS):
        if ";" in v:
            parts = [p.strip() for p in v.split(";") if p.strip()]
            ok = sum(1 for p in parts if len(p.split()) >= 2)
            return round(min(0.75 + 0.06 * ok, 0.95), 2)
        # Tên đầy đủ (≥2 từ, không có số)
        if len(v.split()) >= 2 and not _RE_DIGIT.search(v):
            return 0.90
        _RE_ANON = re.compile(r"[A-ZĐĂÂÊÔƯĐ]\d+", re.UNICODE)
        if _RE_ANON.search(v):
            return 0.82
        if len(v) >= 4:
            return 0.75
        return 0.60

    # ── loai_van_ban / tên văn bản ───────────────────────────────────────────
    if any(k in f for k in ("loai_van_ban", "loai_giay_to", "ten_bien_ban", "ten_toa_an",
                             "co_quan", "toa_an", "toi_danh")):
        words = len(v.split())
        if words >= 4:
            return 0.92
        if words >= 2:
            return 0.85
        return 0.75  # 1 từ như "Quyết định" vẫn hợp lệ → 0.75 thay vì 0.70

    # ── các trường nội dung / tóm tắt ───────────────────────────────────────
    return 0.85 if len(v.split()) >= 3 else (0.78 if len(v.split()) >= 2 else 0.65)

# ── Normalize helpers (O1 + O2) ──────────────────────────────────────────────
_RE_DATE_NORMALIZE = [
    (re.compile(r"(\d{1,2})\s*-\s*(\d{1,2})\s*-\s*(\d{4})"), r"\1/\2/\3"),
    (re.compile(r"ngày\s+(\d{1,2})\s+tháng\s+(\d{1,2})\s+năm\s+(\d{4})", re.I), r"\1/\2/\3"),
]
_RE_SOHIEU_NORMALIZE = re.compile(r"\s*[–—]\s*")

def _normalize_value(value: Any, fname: str) -> Any:
    if value is None or not isinstance(value, str):
        return value
    v, f = value.strip(), fname.lower()
    if any(k in f for k in ("ngay", "ngày", "date", "thoi_gian", "ngay_ky")):
        for pat, repl in _RE_DATE_NORMALIZE:
            v = pat.sub(repl, v)
    if any(k in f for k in ("so_hieu", "so_hop_dong", "so_ban_an")):
        v = _RE_SOHIEU_NORMALIZE.sub("-", v)
    return v

def score_extraction(extracted: dict, sc: Type[BaseModel]) -> Tuple[dict, float, float]:
    scored, confs = {}, []
    for fname in sc.model_json_schema().get("properties", {}):
        raw_val  = extracted.get(fname)
        val      = _normalize_value(raw_val, fname)
        conf     = compute_field_confidence(val, fname)
        is_null  = (val is None or str(val).strip() == "")
        scored[fname] = {
            "value":      val,
            "confidence": conf if not is_null else None,
            "flagged":    False if is_null else (conf < CFG.low_confidence_threshold),
            "missing":    is_null,
        }
        if not is_null:
            confs.append(conf)

    total_fields  = len(scored)
    filled_fields = sum(1 for v in scored.values() if not v["missing"])
    fill_rate     = round(filled_fields / total_fields, 3) if total_fields else 0.0
    overall_conf  = round(sum(confs) / len(confs), 3) if confs else 0.0
    return scored, overall_conf, fill_rate
# ── Input parser ──────────────────────────────────────────────────────────────

def merge_pages_text(doc: dict, max_pages: int = 3) -> str:
    """
    Gop text tu nhieu trang. Thu tu uu tien:
    1. pages list (cau truc multi-page truc tiep)
    2. module2_output.pages list
    3. module2_output text keys (raw_text, text, ocr_text, content, extracted_text)
    4. ground_truth_text
    5. raw_text top-level  [THEM MOI]
    6. ocr_text / content / extracted_text top-level  [THEM MOI]
    7. text fallback cuoi
    """
    # 1. pages list truc tiep tren doc
    if isinstance(doc.get("pages"), list) and doc["pages"]:
        texts = []
        for page in doc["pages"][:max_pages]:
            t = (page.get("text") or page.get("raw_text") or "").strip()
            if t:
                texts.append(t)
        if texts:
            return "\n\n".join(texts)

    # 2-3. module2_output
    m2 = doc.get("module2_output")
    if isinstance(m2, dict) and m2:
        if isinstance(m2.get("pages"), list) and m2["pages"]:
            texts = []
            for page in m2["pages"][:max_pages]:
                t = (page.get("text") or page.get("raw_text") or "").strip()
                if t:
                    texts.append(t)
            if texts:
                return "\n\n".join(texts)
        for key in ("raw_text", "text", "ocr_text", "content", "extracted_text"):
            val = (m2.get(key) or "").strip()
            if val:
                return val

    # 4. ground_truth_text
    gt = (doc.get("ground_truth_text") or "").strip()
    if gt:
        return gt

    # 5. raw_text top-level  [THEM MOI]
    raw = (doc.get("raw_text") or "").strip()
    if raw:
        return raw

    # 6. Alias tu Module 2  [THEM MOI]
    for key in ("ocr_text", "content", "extracted_text"):
        val = (doc.get(key) or "").strip()
        if val:
            return val

    # 7. text fallback cuoi
    return (doc.get("text") or "").strip()


def parse_input_doc(doc: dict) -> Tuple[str, str, str]:
    doc_id = (
        doc.get("document_id")
        or doc.get("sample_id")
        or doc.get("image_name", "Unknown")
    )
    raw = merge_pages_text(doc, max_pages=3)
    return str(doc_id), str(raw), doc.get("schema_type", "")

_DOCID_PREFIX_SCHEMA: List[Tuple[str, str]] = [
    ("toa_an",      "VAN_BAN_TOA_AN"),
    ("bien_ban",    "BIEN_BAN"),
    ("dieu_tra",    "VAN_BAN_DIEU_TRA"),
    ("hanh_chinh",  "QUYET_DINH_HANH_CHINH"),
    ("bao_hiem",    "VAN_BAN_BAO_HIEM"),
    ("don_tu",      "DON_TU_CAM_KET"),
    ("cam_ket",     "DON_TU_CAM_KET"),
    ("hop_dong",    "HOP_DONG"),
    ("dan_su",      "HOP_DONG"),
]

def resolve_schema(doc_id: str, raw_text: str, schema_hint: str) -> Tuple[str, Type[BaseModel]]:
    # Ưu tiên 1: hint tường minh từ metadata
    if schema_hint and schema_hint in SCHEMA_REGISTRY:
        return schema_hint, SCHEMA_REGISTRY[schema_hint]

    # Ưu tiên 2: detect từ text
    name = auto_detect_schema(raw_text)

    # Ưu tiên 3: nếu fallback DON_TU_CAM_KET (không chắc), thử doc_id prefix
    if name == "DON_TU_CAM_KET":
        did_lower = doc_id.lower()
        for prefix, mapped in _DOCID_PREFIX_SCHEMA:
            if prefix in did_lower:
                logger.debug(f"[{doc_id}] prefix-hint: {name} → {mapped}")
                name = mapped
                break

    return name, SCHEMA_REGISTRY.get(name, DonTuCamKetSchema)

def build_result(doc_id: str, schema_name: str, raw_output: str) -> dict:
    parsed = clean_and_parse(raw_output, doc_id)
    if not parsed:
        return {"document_id": doc_id, "schema_used": schema_name,
                "extracted_dynamic_data": None, "confidence_overall": 0.0,
                "error": "json_parse_failed", "raw_output_sample": raw_output[:300]}
    summary = parsed.pop("summary", "") or ""     # ← thêm
    scored, overall, fill_rate = score_extraction(parsed, SCHEMA_REGISTRY[schema_name])
    return {"document_id": doc_id, "schema_used": schema_name,
            "summary": summary,                   # ← thêm
            "extracted_dynamic_data": scored,
            "confidence_overall": overall,
            "fill_rate": fill_rate, "error": None}

logger.info("Helpers loaded")

In [18]:
# ══════════════════════════════════════════════════════════════════════════════
# DIRECT API — Dùng cho debug 1 doc hoặc retry khi batch fail
# ══════════════════════════════════════════════════════════════════════════════

_GEN_CFG = types.GenerateContentConfig(
    response_mime_type="application/json",
    max_output_tokens=CFG.max_output_tokens,
    temperature=0.0,
)

def extract_single(doc: dict, retry: bool = False) -> dict:
    doc_id, raw_text, hint = parse_input_doc(doc)
    # Initialize fill_rate to a default value
    fill_rate = 0.0
    if not raw_text.strip():
        logger.warning(f"[{doc_id}] bỏ qua — văn bản rỗng")
        if fill_rate < CFG.min_fill_rate_warn:
                    logger.warning(
                        f"[{doc_id}] ⚠️  fill={fill_rate:.0%} — "
                        f"có thể là trang nội dung thiếu header. "
                        f"Kiểm tra lại ảnh gốc hoặc thêm schema_hint."
                    )
        return {"document_id": doc_id, "schema_used": "UNKNOWN",
                "extracted_dynamic_data": None, "confidence_overall": 0.0,
                "fill_rate": 0.0, "error": "empty_text"}
    schema_name, schema_class = resolve_schema(doc_id, raw_text, hint)
    last_raw = ""
    for attempt in range(1, CFG.max_retries + 1):
        try:
            prompt   = build_prompt(schema_class, raw_text, retry=(attempt > 1 or retry))
            t0       = time.time()
            resp     = client.models.generate_content(
                model=CFG.model_name, contents=prompt, config=_GEN_CFG
            )
            last_raw = resp.text or ""
            elapsed  = time.time() - t0
            parsed   = clean_and_parse(last_raw, doc_id)
            if parsed:
                    summary = parsed.pop("summary", "") or ""   # ← lấy ra trước khi score
                    scored, overall, fill_rate = score_extraction(parsed, schema_class)
                    # fill rate warn
                    if fill_rate < CFG.min_fill_rate_warn:
                        logger.warning(
                            f"[{doc_id}] ⚠️  fill={fill_rate:.0%} — "
                            f"có thể là trang nội dung thiếu header."
                        )
                    return {
                        "document_id":            doc_id,
                        "schema_used":            schema_name,
                        "summary":                summary,
                        "extracted_dynamic_data": scored,
                        "confidence_overall":     overall,
                        "fill_rate":              fill_rate,
                        "error":                  None,
                    }
            logger.warning(f"[{doc_id}] ⚠️  parse fail lần {attempt}")
        except Exception as e:
            logger.error(f"[{doc_id}] ❌ lần {attempt}: {e}")
            time.sleep(CFG.retry_delay * attempt)
    return {"document_id":doc_id,"schema_used":schema_name,
            "extracted_dynamic_data":None,"confidence_overall":0.0,
            "error":"failed_after_retries","raw_output_sample":last_raw[:300]}

logger.info("✅ Direct API helper ready")

In [19]:
# =============================================================================
# FIX 2 — Batch API (Cell 7 trong notebook)
# Thay thế toàn bộ: _make_batch_request, _poll_batch,
# _collect_batch_results, run_batch_pipeline
#
# Root cause bug cũ:
#   client.batches.create(src=[{"custom_id": ..., "contents": ..., "config": ...}])
#   → Pydantic ValidationError vì InlinedRequest không có field custom_id
#
# Fix:
#   • Dùng types.InlinedRequest(contents=..., config=..., metadata={"id": doc_id})
#     metadata là dict[str,str] hợp lệ, được trả về trong InlinedResponse.metadata
#   • Kết quả lấy qua batch.dest.inlined_responses (không có list_results)
#   • Vẫn giữ nguyên fallback direct API khi batch fail
# =============================================================================

def _make_batch_request(doc: dict) -> Optional[Tuple[str, str, types.InlinedRequest]]:
    """
    Tạo 1 InlinedRequest cho Batch API.
    Trả về (doc_id, schema_name, request) hoặc None nếu doc rỗng.
    """
    doc_id, raw_text, hint = parse_input_doc(doc)
    if not raw_text.strip():
        logger.warning(f"[{doc_id}] bỏ qua — văn bản rỗng")
        return None

    schema_name, schema_class = resolve_schema(doc_id, raw_text, hint)
    prompt = build_prompt(schema_class, raw_text)

    # metadata={"id": doc_id} là cách duy nhất truyền custom ID
    # vì InlinedRequest không có field custom_id
    request = types.InlinedRequest(
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            max_output_tokens=CFG.max_output_tokens,
            temperature=0.0,
        ),
        metadata={"id": doc_id, "schema": schema_name},
    )
    return doc_id, schema_name, request


def _poll_batch(batch_name: str) -> Any:
    """Poll đến khi batch SUCCEEDED hoặc FAILED/CANCELLED. Return batch object."""
    deadline = time.time() + CFG.batch_timeout_sec
    while time.time() < deadline:
        batch = client.batches.get(name=batch_name)
        state = str(batch.state)
        if "SUCCEEDED" in state:
            return batch
        if "FAILED" in state or "CANCELLED" in state:
            logger.error(f"Batch {batch_name} → {state}")
            return batch
        logger.info(f"  ⏳ {batch_name} | {state} | chờ {CFG.batch_poll_interval}s...")
        time.sleep(CFG.batch_poll_interval)
    logger.error(f"Batch {batch_name} timeout sau {CFG.batch_timeout_sec}s")
    return client.batches.get(name=batch_name)


def _collect_batch_results(
    batch: Any,
    id_to_doc: Dict[str, dict],
    id_to_schema: Dict[str, str],
) -> List[dict]:
    """
    Thu kết quả từ batch đã SUCCEEDED.
    Kết quả nằm trong batch.dest.inlined_responses (không phải list_results).
    doc_id được khôi phục từ InlinedResponse.metadata["id"].
    """
    results: List[dict] = []
    failed_ids: List[str] = []

    inlined = getattr(getattr(batch, "dest", None), "inlined_responses", None) or []
    if not inlined:
        logger.warning(f"Batch {batch.name}: dest.inlined_responses rỗng")
        return results

    for resp in inlined:
        # Lấy doc_id từ metadata mà ta đã nhét vào lúc tạo request
        meta = resp.metadata or {}
        doc_id = meta.get("id", "")
        schema_name = meta.get("schema", "")

        if not doc_id or doc_id not in id_to_doc:
            logger.error(f"metadata.id '{doc_id}' không hợp lệ — bỏ qua")
            continue

        # Nếu schema_name bị mất trong metadata, resolve lại từ doc gốc
        if not schema_name:
            _, raw_text, hint = parse_input_doc(id_to_doc[doc_id])
            schema_name, _ = resolve_schema(doc_id, raw_text, hint)

        # Kiểm tra lỗi cấp response
        if resp.error:
            logger.error(f"[{doc_id}] lỗi từ API: {resp.error}")
            failed_ids.append(doc_id)
            continue

        try:
            raw = resp.response.candidates[0].content.parts[0].text or ""
            results.append(build_result(doc_id, schema_name, raw))
        except Exception as e:
            logger.error(f"[{doc_id}] lỗi đọc response: {e}")
            failed_ids.append(doc_id)

    # Retry direct những doc bị lỗi khi collect
    for fid in failed_ids:
        if fid in id_to_doc:
            logger.info(f"[{fid}] retry direct (collect error)")
            results.append(extract_single(id_to_doc[fid]))

    return results


def run_batch_pipeline(documents: List[dict]) -> List[dict]:
    """
    Main pipeline:
      1. Chia documents thành batches (batch_size doc/batch)
      2. Submit tất cả batches dùng types.InlinedRequest — không còn lỗi custom_id
      3. Poll song song tất cả batches
      4. Collect qua batch.dest.inlined_responses, retry direct nếu cần
    """
    total = len(documents)
    n_batches = math.ceil(total / CFG.batch_size)
    logger.info(f"Tổng {total} doc → {n_batches} batch × {CFG.batch_size} doc/batch")

    # ── Bước 1: Submit tất cả batches ────────────────────────────────────────
    # submitted: list of (batch_name_or_None, chunk_docs, id_to_schema_map)
    submitted: List[Tuple[Optional[str], List[dict], Dict[str, str]]] = []

    for i in range(n_batches):
        chunk = documents[i * CFG.batch_size : (i + 1) * CFG.batch_size]

        requests: List[types.InlinedRequest] = []
        id_to_schema: Dict[str, str] = {}

        for doc in chunk:
            result = _make_batch_request(doc)
            if result is None:
                continue
            doc_id, schema_name, req = result
            requests.append(req)
            id_to_schema[doc_id] = schema_name

        if not requests:
            submitted.append((None, chunk, {}))
            continue

        try:
            batch = client.batches.create(
                model=CFG.model_name,
                src=requests,          # list[InlinedRequest] — Pydantic hợp lệ
            )
            logger.info(
                f"  📤 Batch {i+1}/{n_batches} submitted | "
                f"ID: {batch.name} | {len(requests)} requests"
            )
            submitted.append((batch.name, chunk, id_to_schema))
            time.sleep(0.3)           # nhẹ nhàng tránh rate-limit submit
        except Exception as e:
            logger.error(f"  ❌ Batch {i+1} submit lỗi: {e} → fallback direct")
            submitted.append((None, chunk, id_to_schema))

    # ── Bước 2: Poll + collect tất cả batches ────────────────────────────────
    all_results: List[dict] = []

    for batch_name, chunk, id_to_schema in tqdm(submitted, desc="Batches", unit="batch"):
        id_to_doc = {}
        for doc in chunk:
            did, _, _ = parse_input_doc(doc)
            id_to_doc[did] = doc

        if batch_name is None:
            # Batch không submit được → direct fallback
            logger.info(f"  🔄 Direct fallback cho {len(chunk)} doc")
            for doc in tqdm(chunk, desc="  direct", leave=False, unit="doc"):
                all_results.append(extract_single(doc))
            continue

        # Poll batch đến khi xong
        batch = _poll_batch(batch_name)
        state = str(batch.state)

        if "SUCCEEDED" in state:
            batch_results = _collect_batch_results(batch, id_to_doc, id_to_schema)

            # Retry direct những doc không có trong kết quả batch
            returned_ids = {r["document_id"] for r in batch_results}
            missing = [doc for doc in chunk if parse_input_doc(doc)[0] not in returned_ids]
            if missing:
                logger.warning(f"  ⚠️  {len(missing)} doc thiếu trong batch → direct retry")
                for doc in missing:
                    batch_results.append(extract_single(doc))

            all_results.extend(batch_results)
            logger.info(f"  ✅ Batch {batch_name} → {len(batch_results)} kết quả")
        else:
            logger.warning(f"  ⚠️  Batch {batch_name} {state} → direct fallback {len(chunk)} doc")
            # Chạy song song thay vì tuần tự
            with ThreadPoolExecutor(max_workers=5) as pool:
                futures = {pool.submit(extract_single, doc): doc for doc in chunk}
                for fut in tqdm(as_completed(futures), total=len(chunk),
                                desc="  direct fallback", leave=False):
                    try:
                        all_results.append(fut.result())
                    except Exception as e:
                        doc = futures[fut]
                        did, _, _ = parse_input_doc(doc)
                        logger.error(f"[{did}] concurrent fallback lỗi: {e}")

    return all_results


logger.info("✅ Batch pipeline (fixed) ready")

In [20]:
# ══════════════════════════════════════════════════════════════════════════════
# ⚙️  BƯỚC 1 — Debug 1 document (xác nhận API + prompt hoạt động)
# ══════════════════════════════════════════════════════════════════════════════

with open(CFG.input_file, "r", encoding="utf-8") as f:
    _all_docs = json.load(f)
if isinstance(_all_docs, dict):
    _all_docs = [_all_docs]

_doc  = _all_docs[0]
_did, _raw, _hint = parse_input_doc(_doc)
_sname, _scls     = resolve_schema(_did, _raw, _hint)
_prompt           = build_prompt(_scls, _raw)

print(f"Doc ID   : {_did}")
print(f"Schema   : {_sname}")
print(f"Raw text : {_raw[:300]}...")
print(f"Prompt chars: {len(_prompt)} | truncated text chars: {len(smart_truncate(_raw, CFG.max_input_chars))}")
print("\n" + "─"*60)

_result = extract_single(_doc)
print(f"\nStatus          : {'✅ OK' if not _result['error'] else '❌ ' + _result['error']}")
print(f"Confidence      : {_result['confidence_overall']:.1%}")
print(f"Schema used     : {_result['schema_used']}")
print("\nExtracted fields:")
if _result["extracted_dynamic_data"]:
    for k, v in _result["extracted_dynamic_data"].items():
        flag = " ⚠️" if v.get("flagged") else ""
        conf_display = f"{v['confidence']:.2f}" if v['confidence'] is not None else "N/A"
        print(f"  {k:35s}: {str(v['value'])[:60]!r}  (conf={conf_display}){flag}")

Doc ID   : eval_sample_01
Schema   : VAN_BAN_TOA_AN
Raw text : TÒA ÁN NHÂN DÂN                 CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM
     TỈNH BN                                   Độc lập - Tự do - Hạnh phúc
Bản án số: 03/2022/DSST
Ngày: 23/11/2022
V/v: Tranh chấp về kiện đòi tài sản

NHÂN DANH
NƯỚC CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM

TÒA ÁN NHÂN DÂN TỈNH BN

- Th...
Prompt chars: 2737 | truncated text chars: 1622

────────────────────────────────────────────────────────────

Status          : ✅ OK
Confidence      : 90.4%
Schema used     : VAN_BAN_TOA_AN

Extracted fields:
  loai_van_ban                       : 'Bản án sơ thẩm'  (conf=0.92)
  so_hieu                            : '03/2022/DSST'  (conf=0.95)
  ngay_ban_hanh                      : '23/11/2022'  (conf=0.92)
  ten_toa_an                         : 'Tòa án nhân dân tỉnh BN'  (conf=0.92)
  vu_viec_tom_tat                    : 'Vụ án tranh chấp về kiện đòi tài sản.'  (conf=0.85)
  tham_phan_chu_toa                  : 'Nguyễn T5Mai

In [21]:
# ══════════════════════════════════════════════════════════════════════════════
# ⚙️  BƯỚC 2 — Main pipeline (chạy toàn bộ, Batch API)
# Chạy cell này SAU KHI debug cell trên OK
# ══════════════════════════════════════════════════════════════════════════════

with open(CFG.input_file, "r", encoding="utf-8") as f:
    documents = json.load(f)
if isinstance(documents, dict):
    documents = [documents]

logger.info(f"Tổng: {len(documents)} doc | batch_size={CFG.batch_size}")
t_start = time.time()

results = run_batch_pipeline(documents)

# ── Thống kê ──────────────────────────────────────────────────────────────────
# ── Thống kê ──────────────────────────────────────────────────────────────────
from collections import Counter

elapsed   = time.time() - t_start
success   = [r for r in results if not r.get("error")]
errors    = [r for r in results if  r.get("error")]
n_total   = len(results)
n_success = len(success)

# Confidence (chỉ tính field có giá trị)
avg_conf  = sum(r["confidence_overall"] for r in success) / n_success if success else 0
high_conf = sum(1 for r in success if r["confidence_overall"] >= 0.80)
med_conf  = sum(1 for r in success if 0.65 <= r["confidence_overall"] < 0.80)
low_conf  = sum(1 for r in success if r["confidence_overall"] < 0.65)

# Fill rate — tỷ lệ field có dữ liệu thực sự
avg_fill  = sum(r.get("fill_rate", 0) for r in success) / n_success if success else 0

# Tốc độ xử lý
sec_per_doc = elapsed / n_total if n_total else 0
doc_per_min = 60 / sec_per_doc if sec_per_doc else 0

# Schema distribution
schema_dist = Counter(r["schema_used"] for r in results)

print("\n" + "═"*62)
print(f"Hoàn thành   : {n_total}/{len(documents)} doc | {elapsed:.1f}s tổng")
print(f"Tốc độ       : {sec_per_doc:.1f}s/doc | {doc_per_min:.1f} doc/phút")
print(f"Thành công   : {n_success} ({n_success/n_total:.0%}) | Lỗi: {len(errors)}")
print(f"Confidence   : TB={avg_conf:.1%} (chỉ field có data) | ≥80%: {high_conf} | 65–80%: {med_conf} | <65%: {low_conf}")
print(f"Fill rate    : TB={avg_fill:.1%} field có dữ liệu/doc")
print(f"Schema dist  : {dict(schema_dist)}")
print("═"*62)

if errors:
    print(f"\n❌ {len(errors)} doc lỗi:")
    for e in errors[:10]:
        print(f"  [{e['document_id']}] {e['error']} | raw='{e.get('raw_output_sample','')[:60]}'")

# ── Lưu kết quả ───────────────────────────────────────────────────────────────
with open(CFG.output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
logger.info(f"✅ Đã lưu {len(results)} kết quả → {CFG.output_file}")

Batches:   0%|          | 0/3 [00:00<?, ?batch/s]


══════════════════════════════════════════════════════════════
Hoàn thành   : 44/44 doc | 108.9s tổng
Tốc độ       : 2.5s/doc | 24.2 doc/phút
Thành công   : 43 (98%) | Lỗi: 1
Confidence   : TB=84.1% (chỉ field có data) | ≥80%: 41 | 65–80%: 0 | <65%: 2
Fill rate    : TB=70.5% field có dữ liệu/doc
Schema dist  : {'VAN_BAN_TOA_AN': 22, 'DON_TU_CAM_KET': 4, 'QUYET_DINH_HANH_CHINH': 7, 'VAN_BAN_BAO_HIEM': 3, 'VAN_BAN_DIEU_TRA': 2, 'BIEN_BAN': 4, 'UNKNOWN': 1, 'HOP_DONG': 1}
══════════════════════════════════════════════════════════════

❌ 1 doc lỗi:
  [eval_sample_12] empty_text | raw=''


In [22]:
# ══════════════════════════════════════════════════════════════════════════════
# ⚙️  BƯỚC 3 (tuỳ chọn) — Phân tích kết quả, xem doc confidence thấp
# ══════════════════════════════════════════════════════════════════════════════

LOW_CONF_THRESHOLD = 0.60   # Chỉnh ngưỡng tuỳ ý
min_fill_rate_warn: float = 0.40
low_conf_docs = [r for r in results if r["confidence_overall"] < LOW_CONF_THRESHOLD and not r["error"]]
flagged_fields = []
for r in results:
    if r["extracted_dynamic_data"]:
        for fname, fdata in r["extracted_dynamic_data"].items():
            if fdata.get("flagged"):
                flagged_fields.append({"doc_id": r["document_id"], "field": fname,
                                       "value": fdata.get("value"), "conf": fdata.get("confidence"),
                                       "warn": fdata.get("warning","")})

print(f"Doc confidence < {LOW_CONF_THRESHOLD:.0%}: {len(low_conf_docs)}")
for r in low_conf_docs[:5]:
    print(f"  [{r['document_id']}] schema={r['schema_used']} conf={r['confidence_overall']:.1%}")

print(f"\nTổng field bị flag: {len(flagged_fields)}")
for f in flagged_fields[:10]:
    warn = f" ← {f['warn']}" if f['warn'] else ""
    print(f"  [{f['doc_id']}] {f['field']}: {str(f['value'])[:40]!r} (conf={f['conf']:.2f}){warn}")

# Xuất riêng low-confidence để review thủ công
low_conf_out = CFG.output_file.replace(".json","_low_conf.json")
with open(low_conf_out, "w", encoding="utf-8") as f:
    json.dump(low_conf_docs, f, ensure_ascii=False, indent=2)
print(f"\n📄 Low-confidence docs → {low_conf_out}")

Doc confidence < 60%: 2
  [eval_sample_02] schema=DON_TU_CAM_KET conf=0.0%
  [eval_sample_07] schema=QUYET_DINH_HANH_CHINH conf=0.0%

Tổng field bị flag: 0

📄 Low-confidence docs → /content/drive/MyDrive/VNDigitize/UIE_Extraction_Results_batch_low_conf.json
